In [1]:
import os
%pwd

'/Users/kiranprasadjp/Documents/Pros/NLP/hf-summarizer/research'

In [2]:
os.chdir('../')
%pwd

'/Users/kiranprasadjp/Documents/Pros/NLP/hf-summarizer'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainingConfig:
    rootDir: Path
    dataPath: Path
    modelCkpt: Path
    numTrainEpoch: int
    warmupSteps: int
    perDeviceTrainBatchSize: int
    weightDecay: float
    loggingSteps: int
    evaluationStratergy: str
    evalSteps: int
    saveSteps: int
    gradientAccumulationSteps: int

In [4]:
from src.text_summarizer.constants import *
from src.text_summarizer.utils.common import readYaml, createDir

In [5]:
class ConfigurationManager:
    def __init__(self, config_path=CONFIG_FILE_PATH, params_path= PARAMS_FILE_PATH):
        self.config=readYaml(config_path)
        self.params=readYaml(params_path)

        createDir([self.config.artifactsRoot])

    def getModelTrainingConfig(self)-> ModelTrainingConfig:
        config=self.config.modelTrainer
        params= self.params.TrainingArgumets
        createDir([config.rootDir])
        modelTrainingConfig= ModelTrainingConfig(
            rootDir= config.rootDir,
            dataPath= config.dataPath,
            modelCkpt= config.modelCkpt,
            numTrainEpoch= params.numTrainEpoch,
            warmupSteps= params.warmupSteps,
            perDeviceTrainBatchSize= params.perDeviceTrainBatchSize,
            weightDecay= params.weightDecay,
            loggingSteps= params.loggingSteps,
            evaluationStratergy= params.evaluationStratergy,
            evalSteps= params.evalSteps,
            saveSteps= params.saveSteps,
            gradientAccumulationSteps= params.gradientAccumulationSteps,

        )

        return modelTrainingConfig

            

In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch
from datasets import load_from_disk

/Users/kiranprasadjp/Documents/Pros/NLP/hf-summarizer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class ModelTraining:
    def __init__(self, config= ModelTrainingConfig):
        self.config=config
        
    def train(self):
        device= "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer= AutoTokenizer.from_pretrained(self.config.modelCkpt)
        modelPegasus= AutoModelForSeq2SeqLM.from_pretrained(self.config.modelCkpt).to(device)
        seq2SeqDataCollator = DataCollatorForSeq2Seq(tokenizer,model= modelPegasus)

        # loading data
        dfSamsumPt = load_from_disk(self.config.dataPath)

        # the dataset already includes tokenized fields from transformation stage;
        # Trainer expects `input_ids` and `attention_mask` names, so rename columns
        dfSamsumPt = dfSamsumPt.rename_column("inputIds", "input_ids")
        dfSamsumPt = dfSamsumPt.rename_column("attentionMask", "attention_mask")

        trainingArgs= TrainingArguments(
            output_dir=self.config.rootDir,
            num_train_epochs= self.config.numTrainEpoch,
            warmup_steps= self.config.warmupSteps,
            per_device_train_batch_size= self.config.perDeviceTrainBatchSize,
            per_device_eval_batch_size=1,
            weight_decay= self.config.weightDecay,
            logging_steps= self.config.loggingSteps,
            # evaluation_stratergy = self.config.evaluationStratergy,
            eval_steps= self.config.evalSteps,
            save_steps= 1,
            gradient_accumulation_steps= self.config.gradientAccumulationSteps,
        )
        trainer=Trainer(model= modelPegasus, args=trainingArgs,processing_class=tokenizer,
                        data_collator=seq2SeqDataCollator, train_dataset=dfSamsumPt['train'],
                        eval_dataset=dfSamsumPt['validation'])
        trainer.train()

        modelPegasus.save_pretrained(os.path.join(self.config.rootDir,'pegasus_samsum_model'))
        tokenizer.save_pretrained(os.path.join(self.config.rootDir,'tokenizer'))

In [18]:
config = ConfigurationManager()
modelTrainingConfig = config.getModelTrainingConfig()
modelTraining= ModelTraining(config=modelTrainingConfig)
modelTraining.train()

[2026-03-05 14:14:33,072: INFO: common: YAML file:config/config.yaml loaded successfully]
[2026-03-05 14:14:33,077: INFO: common: YAML file:params.yaml loaded successfully]
[2026-03-05 14:14:33,078: INFO: common: Directory Created at: artifacts]
[2026-03-05 14:14:33,079: INFO: common: Directory Created at: artifacts/model_trainer]
[2026-03-05 14:14:33,208: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-05 14:14:33,238: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-03-05 14:14:33,291: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-05 14:14:33,323: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/mod

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 58701.44it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Con

[2026-03-05 14:14:38,917: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-05 14:14:38,943: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/generation_config.json "HTTP/1.1 200 OK"]


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


RuntimeError: MPS backend out of memory (MPS allocated: 10.60 GiB, other allocations: 9.54 GiB, max allowed: 20.13 GiB). Tried to allocate 4.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).